In [2]:
import torch

# 检查 CUDA(NVIDIA GPU) 后端是否可用
print(torch.cuda.is_available())
# 检查 MPS(Apple Silicon GPU) 后端是否可用
print(torch.backends.mps.is_available())

# 只有在 CUDA 可用时才查询设备名，否则会报错
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# 当前默认使用的 GPU 设备索引(从 0 开始)
print(torch.cuda.current_device())
# 第 0 张 GPU 的名称
print(torch.cuda.get_device_name(0))
# PyTorch 版本，+cu130 表示编译时绑定的 CUDA 版本为 13.0
print(torch.__version__)


True
False
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
0
NVIDIA GeForce RTX 4060 Laptop GPU
2.12.0+cu130


In [6]:


'''
用本地缓存的 Qwen2.5-1.5B-Instruct 模型做一次单轮对话推理（chat completion），
即给模型一句提问，让它生成一段回答文本，最终结果保存在 response 变量里
'''
# 从 ModelScope(魔搭) 加载因果语言模型与对应的分词器



from modelscope import AutoModelForCausalLM, AutoTokenizer

# 本地的模型目录(提前下载好的 Qwen2.5-1.5B-Instruct)
model_name = "/home/tommy/.cache/modelscope/hub/models/Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",       # 自动采用模型配置中的权重精度(通常为 bfloat16/float16)
    device_map="auto"         # 自动把各层分配到可用的 GPU/CPU 上
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 组织对话消息：system 设定角色，user 为用户的提问
prompt = "who are you."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

# 套用聊天模板，把消息列表拼成模型期望的纯文本
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,             # 先只返回字符串，不做分词
    add_generation_prompt=True  # 末尾追加 assistant 起始标记，提示模型开始回答
)

# 分词为 PyTorch 张量，并搬到模型所在的设备(如 cuda:0)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 自回归生成，最多新产生 512 个 token(返回值包含输入 + 新生成的 token)
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

# 按输入长度切片，只保留新生成的部分，去掉回显的 prompt
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码回文本，skip_special_tokens 去掉 <|im_end|> 等特殊标记
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

I am Qwen, an AI language model developed by Alibaba Cloud. I was trained on large amounts of text data to understand and generate human-like responses. My purpose is to assist users in generating creative writing, scientific articles, poetry, and more. Please feel free to ask me any questions or let me know how I can help you today!
